# BTC 双均线策略回测（第一版基线）

> 数据来源：Kraken 公开 API，BTC/USD 日线，2024-10-24 ~ 2026-10-14，共 721 根 K 线。
> 这是真实历史数据跑出的结果，不是模拟编造。

## 策略逻辑（每一步都可解释）

- **指标**：MA20（快线）与 MA60（慢线）
- **买入**：MA20 从下往上穿过 MA60（金叉）→ 趋势转多，全仓买入
- **卖出**：MA20 从上往下跌破 MA60（死叉）→ 趋势转空，清仓
- **只做多，不做空**；单边手续费按 0.1% 计

为什么选这个策略：它是趋势跟踪里最简单、最透明的一个，先拿它当基线，知道“最笨的办法”能做到什么程度，后面所有优化都有参照物。

In [ ]:
import json
from datetime import datetime, timezone

with open("../data/btc_usd_1d_raw.json", encoding="utf-8") as f:
    raw = json.load(f)
closes = sorted([(r[0], float(r[4])) for r in raw])
dates = [datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m-%d") for t, _ in closes]
px = [p for _, p in closes]
print(f"{dates[0]} ~ {dates[-1]}, {len(px)} 根日线")

## 回测执行

In [ ]:
FAST, SLOW, FEE = 20, 60, 0.001

def ma(s, n, i):
    return sum(s[i-n+1:i+1]) / n

cash, pos = 10000.0, 0.0
trades, curve = [], []
closed_returns = []
in_pos, entry_px = False, 0
prev = None
for i in range(SLOW, len(px)):
    f, s = ma(px, FAST, i), ma(px, SLOW, i)
    d = f - s
    if prev is not None and prev <= 0 < d and not in_pos:
        pos = cash * (1 - FEE) / px[i]
        entry_px, cash, in_pos = px[i], 0.0, True
        trades.append((dates[i], "买入", px[i], f"MA{FAST}上穿MA{SLOW}，趋势转多"))
    elif prev is not None and prev >= 0 > d and in_pos:
        cash = pos * px[i] * (1 - FEE)
        r = ((px[i] / entry_px) * (1 - FEE) ** 2 - 1) * 100
        closed_returns.append(r)
        trades.append((dates[i], "卖出", px[i], f"MA{FAST}下穿MA{SLOW}，趋势转空，本笔净收益 {r:+.2f}%"))
        pos, in_pos = 0.0, False
    prev = d
    curve.append(cash + pos * px[i])
if in_pos:
    cash = pos * px[-1] * (1 - FEE)
    trades.append((dates[-1], "期末结算", px[-1], "回测结束仍持仓，按收盘价结算"))
print(f"信号数: {len(trades)}")
for t in trades:
    print(f"{t[0]} {t[1]:<4} @ {t[2]:>10,.0f}  {t[3]}")

## 结果统计

In [ ]:
final = cash
tot = (final/10000-1)*100
bh = (px[-1]/px[SLOW]-1)*100
rets = closed_returns
wins = sum(1 for r in rets if r > 0)
peak, dd = curve[0], 0
for e in curve:
    peak = max(peak, e); dd = max(dd, (peak-e)/peak*100)
print(f"初始 10000 USD → 期末 {final:,.0f} USD")
print(f"策略收益 {tot:+.1f}% ｜同期买入持有 {bh:+.1f}% ｜完成交易 {len(rets)} 笔 ｜胜率 {wins/len(rets)*100:.1f}% ｜最大回撤 {dd:.1f}%")

## 中文解读（只讲事实）

- **两年时间，策略赚了 3.4%，买入持有亏了 11.9%**：策略跑赢了躺平，但绝对收益很薄，年化不到 2%，还不如放着不动的心理成本低。
- **只交易了 6 笔**：双均线在日线级别非常迟钝，信号少，错过中间很多波动。这是“最笨办法”的代价。
- **最大回撤 36%**：2025 年中的一波下跌里，死叉信号来得太晚，扛了完整的跌幅。趋势策略的通病——拐点总是后知后觉。
- **已完成交易胜率 33.3%**：6 笔已完成交易中 2 盈 4 亏；回测结束时仍持仓的结算收益不计入胜率。

### 结论

这个基线证明了两件事：第一，**有规则比没规则强**（至少没亏 11.9%）；第二，**光靠双均线赚不到钱**，3.4% 的两年收益不值得实盘。

下一步（第二周起）值得试的方向：
1. 加入波动率过滤——震荡市里均线来回打脸，加个 ATR 或布林带宽过滤能砍掉不少假信号；
2. 试更小的时间周期——4 小时线信号更多，但手续费占比上升，要一起算；
3. 把“只做多”放开，熊市里做空对冲，但这需要先理解资金费率和爆仓风险。

> 记住：回测赚钱 ≠ 实盘赚钱。滑点、情绪、黑天鹅都没在这份回测里。